# Experiment: TCP Content-Aware Adaptive Batching

Objective:
- `init_plan.md`를 기준으로 TCP 콘텐츠별 배칭 실험 노트북의 baseline 실행 골격을 만든다.
- `static_file`과 `dynamic_stream`에 대해 `immediate`, `fixed_batch`, `heuristic_adaptive`를 먼저 비교한다.
- 다음 단계에서 full fixed sweep, oracle label, `ml_regression_adaptive`를 확장할 수 있도록 함수 경계를 고정한다.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import math
import subprocess
import sys
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import List, Optional, Tuple

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
}

missing = [pkg for module, pkg in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

SEED = 20260309
MSS_BYTES = 1460
FIXED_BATCH_GRID = [512, 2048, 8192, 32768, 65536]
FIXED_FLUSH_GRID = [0.0, 2.0, 5.0, 10.0, 20.0, 50.0]

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 120

OUTPUT_DIR = Path("output/jupyter-notebook")
ASSET_DIR = OUTPUT_DIR / "assets"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ASSET_DIR.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "python": sys.version.split()[0],
    "seed": SEED,
    "output_dir": str(OUTPUT_DIR),
    "assets_dir": str(ASSET_DIR),
}, indent=2))


## Baseline Scope

- 워크로드: `static_file`, `dynamic_stream`
- baseline 정책: `immediate`, `fixed_batch(8192B/10ms)`, `heuristic_adaptive`
- 대표 지표: `latency_mean_ms`, `latency_p95_ms`, `throughput_mbps`, `goodput_bytes`, `flush_count`, `mean_batch_size_bytes`, `staleness_penalty`
- 본 노트북은 대표 시나리오 비교까지 먼저 고정하고, full sweep과 ML 학습은 다음 셀 추가로 확장한다.


In [ ]:
@dataclass(frozen=True)
class WorkloadConfig:
    content_type: str
    file_size_bytes: Optional[int] = None
    chunk_size_bytes: Optional[int] = None
    message_size_bytes: Optional[int] = None
    interarrival_ms: Optional[float] = None
    change_rate: str = "low"
    freshness_budget_ms: Optional[float] = None
    rtt_ms: float = 10.0
    bandwidth_mbps: float = 20.0
    delayed_ack_ms: float = 10.0
    generation_gap_ms: float = 0.05
    sample_messages: int = 240

@dataclass(frozen=True)
class PolicyConfig:
    name: str
    batch_bytes: Optional[int] = None
    flush_interval_ms: Optional[float] = None

REFERENCE_FIXED = PolicyConfig("fixed_batch", 8192, 10.0)

REFERENCE_CASES = [
    WorkloadConfig(
        content_type="static_file",
        file_size_bytes=16 * 1024 * 1024,
        chunk_size_bytes=4096,
        rtt_ms=30,
        bandwidth_mbps=20,
        delayed_ack_ms=40,
    ),
    WorkloadConfig(
        content_type="dynamic_stream",
        message_size_bytes=1200,
        interarrival_ms=10,
        change_rate="high",
        freshness_budget_ms=25,
        rtt_ms=50,
        bandwidth_mbps=5,
        delayed_ack_ms=40,
        sample_messages=300,
    ),
]

pd.DataFrame([asdict(cfg) for cfg in REFERENCE_CASES]).fillna("-")


In [ ]:
def scenario_id(cfg: WorkloadConfig) -> str:
    if cfg.content_type == "static_file":
        return f"static_{cfg.chunk_size_bytes}_{cfg.rtt_ms}_{cfg.bandwidth_mbps}_{cfg.delayed_ack_ms}"
    return f"stream_{cfg.message_size_bytes}_{cfg.interarrival_ms}_{cfg.change_rate}_{cfg.rtt_ms}_{cfg.bandwidth_mbps}_{cfg.freshness_budget_ms}"

def generate_workload(cfg: WorkloadConfig) -> pd.DataFrame:
    local_rng = np.random.default_rng(SEED + sum(ord(ch) for ch in scenario_id(cfg)) % 10000)
    if cfg.content_type == "static_file":
        count = int(math.ceil(cfg.file_size_bytes / cfg.chunk_size_bytes))
        payloads = np.full(count, cfg.chunk_size_bytes, dtype=int)
        payloads[-1] -= max(0, count * cfg.chunk_size_bytes - cfg.file_size_bytes)
        event_time_ms = np.arange(count, dtype=float) * cfg.generation_gap_ms
        freshness = np.full(count, np.inf, dtype=float)
    else:
        jitter = 0.10 if cfg.change_rate == "low" else 0.25
        gaps = local_rng.normal(cfg.interarrival_ms, max(cfg.interarrival_ms * jitter, 0.5), size=cfg.sample_messages)
        gaps = np.clip(gaps, 1.0, None)
        event_time_ms = np.cumsum(gaps) - gaps[0]
        payload_noise = local_rng.uniform(0.8, 1.2, size=cfg.sample_messages)
        payloads = np.maximum(64, np.rint(cfg.message_size_bytes * payload_noise).astype(int))
        freshness = np.full(cfg.sample_messages, cfg.freshness_budget_ms, dtype=float)
    return pd.DataFrame({
        "event_time_ms": event_time_ms,
        "payload_bytes": payloads,
        "freshness_budget_ms": freshness,
    })

def snap(value: float, grid: List[float]) -> float:
    return min(grid, key=lambda candidate: abs(candidate - value))

def resolve_policy(cfg: WorkloadConfig, policy: PolicyConfig) -> Tuple[int, float]:
    if policy.name == "immediate":
        return 1, 0.0
    if policy.name == "fixed_batch":
        return int(policy.batch_bytes), float(policy.flush_interval_ms)
    if cfg.content_type == "static_file":
        bdp = (cfg.bandwidth_mbps * 1_000_000 / 8.0) * (cfg.rtt_ms / 1000.0)
        batch = max(4 * MSS_BYTES, 0.5 * bdp)
        flush_ms = min(0.5 * cfg.rtt_ms, cfg.delayed_ack_ms)
    else:
        batch = min(2 * MSS_BYTES, cfg.message_size_bytes * 4.0)
        flush_ms = min(0.5 * cfg.freshness_budget_ms, cfg.interarrival_ms * 1.5)
        if cfg.change_rate == "high":
            batch *= 0.5
            flush_ms *= 0.5
    return int(max(1, snap(batch, FIXED_BATCH_GRID))), float(max(0.0, snap(flush_ms, FIXED_FLUSH_GRID)))

def run_simulation(cfg: WorkloadConfig, policy: PolicyConfig) -> dict:
    events = generate_workload(cfg)
    batch_target, flush_limit = resolve_policy(cfg, policy)
    arrivals, freshness, payloads = [], [], []
    queue_bytes = 0
    batch_start = None
    latencies, stale, batch_sizes = [], [], []
    total_payload, flush_count = 0, 0
    first_arrival = float(events["event_time_ms"].iloc[0]) if not events.empty else 0.0
    last_completion = first_arrival

    def flush(now: float) -> None:
        nonlocal arrivals, freshness, payloads, queue_bytes, batch_start, total_payload, flush_count, last_completion
        if not payloads:
            return
        payload = int(sum(payloads))
        tx_time = (payload * 8.0) / (cfg.bandwidth_mbps * 1_000_000.0) * 1000.0
        ack_penalty = min(cfg.delayed_ack_ms, 0.25 * cfg.rtt_ms) if payload < MSS_BYTES else 0.0
        completion = now + tx_time + (cfg.rtt_ms / 2.0) + ack_penalty
        for arrival, budget in zip(arrivals, freshness):
            latency = completion - arrival
            latencies.append(latency)
            stale.append(max(0.0, latency - budget) if math.isfinite(budget) else 0.0)
        batch_sizes.append(payload)
        total_payload += payload
        flush_count += 1
        last_completion = completion
        arrivals, freshness, payloads = [], [], []
        queue_bytes = 0
        batch_start = None

    for row in events.itertuples(index=False):
        t = float(row.event_time_ms)
        if payloads and flush_limit > 0 and batch_start is not None and t - batch_start >= flush_limit:
            flush(batch_start + flush_limit)
        if not payloads:
            batch_start = t
        arrivals.append(t)
        freshness.append(float(row.freshness_budget_ms))
        payloads.append(int(row.payload_bytes))
        queue_bytes += int(row.payload_bytes)
        if flush_limit == 0.0 or queue_bytes >= batch_target:
            flush(t)
    if payloads:
        flush(max(float(events["event_time_ms"].iloc[-1]), batch_start + flush_limit if flush_limit > 0 else float(events["event_time_ms"].iloc[-1])))

    duration = max(last_completion - first_arrival, 1e-6)
    latency_array = np.asarray(latencies, dtype=float)
    stale_array = np.asarray(stale, dtype=float)
    batch_array = np.asarray(batch_sizes, dtype=float)
    return {
        **asdict(cfg),
        "scenario_id": scenario_id(cfg),
        "policy": policy.name,
        "resolved_batch_bytes": batch_target,
        "resolved_flush_interval_ms": flush_limit,
        "latency_mean_ms": float(latency_array.mean()),
        "latency_p95_ms": float(np.quantile(latency_array, 0.95)),
        "throughput_mbps": float((total_payload * 8.0) / duration / 1000.0),
        "goodput_bytes": float(total_payload),
        "flush_count": int(flush_count),
        "mean_batch_size_bytes": float(batch_array.mean()),
        "staleness_penalty": float(stale_array.sum()),
    }

POLICIES = [
    PolicyConfig("immediate"),
    REFERENCE_FIXED,
    PolicyConfig("heuristic_adaptive"),
]

reference_results = pd.DataFrame([
    run_simulation(cfg, policy)
    for cfg in REFERENCE_CASES
    for policy in POLICIES
])

reference_summary = reference_results[[
    "content_type",
    "policy",
    "latency_mean_ms",
    "latency_p95_ms",
    "throughput_mbps",
    "goodput_bytes",
    "flush_count",
    "mean_batch_size_bytes",
    "staleness_penalty",
]].copy()

reference_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), constrained_layout=True)
sns.barplot(data=reference_summary, x="policy", y="throughput_mbps", hue="content_type", ax=axes[0])
axes[0].set_title("Reference Throughput Comparison")
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(data=reference_summary, x="policy", y="latency_p95_ms", hue="content_type", ax=axes[1])
axes[1].set_title("Reference p95 Latency Comparison")
axes[1].tick_params(axis="x", rotation=15)
if axes[1].legend_ is not None:
    axes[1].legend_.remove()

figure_path = ASSET_DIR / "reference_policy_overview.png"
fig.savefig(figure_path, bbox_inches="tight")
plt.show()

print(f"Saved {figure_path}")
reference_summary


## Next Steps

- `fixed_sweep(scenarios)`와 `pick_oracle(sweep_df)` 셀을 추가해 full matrix 실험으로 확장한다.
- `feature_frame`, `RandomForestRegressor` 기반의 `ml_regression_adaptive` 학습 셀을 추가한다.
- heatmap, Pareto scatter, 발표용 CSV export를 `output/jupyter-notebook/assets/` 아래에 정리한다.
- 실험 축이 바뀌면 먼저 [`init_plan.md`](/C:/git/network/init_plan.md)를 수정하고 그 다음 이 노트북 상단 설정을 맞춘다.
